In [ ]:
import pyspark
import pandas as pd
import dxpy
import dxdata
import json
import numpy as np
from bokeh.io import show, output_notebook
from bokeh.layouts import gridplot
import random
import seaborn as sns
import matplotlib.pyplot as plt
output_notebook()

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

In [ ]:
import hail as hl
hl.init(sc=sc, default_reference='GRCh38')
db_name = "base_drug_phenos"
db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database")['id']
url = f"dnax://{db_uri}/base.ht"
full = hl.read_table(url)

In [ ]:
bnf_code_df = pd.read_csv('<NOTEBOOKS_DIR>/data/bnf_drug.csv')
bnf_code_df = bnf_code_df[bnf_code_df['group'] == 'Angiotensin-II Receptor Antagonists']
with open('<NOTEBOOKS_DIR>/data/drug_brand_names_dict.json', 'r') as file:
    brand_drug_dict = json.load(file)
brand_names_list = list(brand_drug_dict.keys())

In [ ]:
read_v2_code_df = pd.read_csv('<NOTEBOOKS_DIR>/data/read_2_drug.csv')
read_v2_code_df = read_v2_code_df[read_v2_code_df['group'] == 'Angiotensin-II Receptor Antagonists']
read_ctv3_code_df = pd.read_csv('<NOTEBOOKS_DIR>/data/ctv3_drug.csv')
read_ctv3_code_df = read_ctv3_code_df[read_ctv3_code_df['group'] == 'Angiotensin-II Receptor Antagonists']
dmd_code_df = pd.read_csv('<NOTEBOOKS_DIR>/data/dmd_drug.csv')
dmd_code_df = dmd_code_df[dmd_code_df['group'] == 'Angiotensin-II Receptor Antagonists']
dmd_code_df['dmd_code'] = dmd_code_df['dmd_code'].astype(str)

In [ ]:
brand_names_lower = hl.literal([name.lower() for name in brand_names_list])

In [ ]:
all_filtered = full.filter(((hl.literal(read_v2_code_df['read_code'].tolist()).contains(full['code'])) & (full['system'] == 'read_2')) 
                      | (hl.any(lambda substring: full['info'].lower().contains(substring), brand_names_lower))
                      | ((hl.literal(bnf_code_df['bnf_code'].tolist()).contains(full['code'])) & (full['system'] == 'bnf'))
                      | ((hl.literal(dmd_code_df['dmd_code'].tolist()).contains(full['code'])) & (full['system'] == 'dmd'))
                      | ((hl.literal(read_ctv3_code_df['read_code'].tolist()).contains(full['code'])) & (full['system'] == 'read_3'))
                     )

In [ ]:
unique_eids = all_filtered.aggregate(hl.agg.collect_as_set(all_filtered.eid))

In [ ]:
full = full.filter(hl.literal(unique_eids).contains(full.eid))

In [ ]:
full = full.cache()

In [ ]:
bnf_code_df = pd.read_csv('<NOTEBOOKS_DIR>/data/all_bnf_drug.csv')
read_v2_code_df = pd.read_csv('<NOTEBOOKS_DIR>/data/all_read_2_drug.csv')
read_ctv3_code_df = pd.read_csv('<NOTEBOOKS_DIR>/data/all_ctv3_drug.csv')
dmd_code_df = pd.read_csv('<NOTEBOOKS_DIR>/data/all_dmd_drug.csv')
dmd_code_df['dmd_code'] = dmd_code_df['dmd_code'].astype(str)

In [ ]:
all_filtered = full.filter(((hl.literal(read_v2_code_df['read_code'].tolist()).contains(full['code'])) & (full['system'] == 'read_2')) 
                      | (hl.any(lambda substring: full['info'].lower().startswith(substring), brand_names_lower))
                      | ((hl.literal(bnf_code_df['bnf_code'].tolist()).contains(full['code'])) & (full['system'] == 'bnf'))
                      | ((hl.literal(dmd_code_df['dmd_code'].tolist()).contains(full['code'])) & (full['system'] == 'dmd'))
                      | ((hl.literal(read_ctv3_code_df['read_code'].tolist()).contains(full['code'])) & (full['system'] == 'read_3'))
                     )

In [ ]:
def get_drug_name(info, brand_drug_dict):
    brand_drug_hl = hl.literal(brand_drug_dict)
    brand_names = hl.literal(list(brand_drug_dict.keys()))
    
    matched_brand = hl.find(
        lambda brand: hl.str(info).lower().contains(brand.lower()),
        brand_names
    )
    
    return hl.if_else(
        hl.is_defined(matched_brand), 
        brand_drug_hl.get(matched_brand), 
        hl.missing(hl.tstr)  
    )

In [ ]:
all_filtered = all_filtered.annotate(
    drug_name=get_drug_name(all_filtered['info'], brand_drug_dict)
)

In [ ]:
missing = all_filtered.filter(hl.is_missing(all_filtered.drug_name))
non_missing = all_filtered.filter(hl.is_defined(all_filtered.drug_name))

In [ ]:
brand_names_list = list(brand_drug_dict.keys())

In [ ]:
read_v2_code_to_drug_dict = dict(zip(read_v2_code_df['read_code'], read_v2_code_df['term']))
read_ctv3_code_to_drug_dict = dict(zip(read_ctv3_code_df['read_code'], read_ctv3_code_df['term']))
dmd_code_to_drug_dict = dict(zip(dmd_code_df['dmd_code'], dmd_code_df['term']))
bnf_code_drug_dict = dict(zip(bnf_code_df['bnf_code'], bnf_code_df['term']))

In [ ]:
read_v2_literal = hl.literal(read_v2_code_to_drug_dict)
dmd_literal = hl.literal(dmd_code_to_drug_dict)
read_ctv3_literal = hl.literal(read_ctv3_code_to_drug_dict)
bnf_literal = hl.literal(bnf_code_drug_dict)

In [ ]:
missing = missing.annotate(
    drug_name = hl.case()
        .when((missing.system == 'read_2') & (read_v2_literal.contains(missing.code)),
              read_v2_literal[missing.code])
        .when((missing.system == 'dmd') & (dmd_literal.contains(missing.code)),
              dmd_literal[missing.code])
        .when((missing.system == 'read_3') & (read_ctv3_literal.contains(missing.code)),
              read_ctv3_literal[missing.code])
        .when((missing.system == 'bnf') & (bnf_literal.contains(missing.code)),
              bnf_literal[missing.code])
        .or_missing()  
)

In [ ]:
all_ht = missing.union(non_missing)

In [ ]:
with open('<NOTEBOOKS_DIR>/data/type_drug_dict.json', 'r') as file:
    n_type_drug_dict = json.load(file)
type_drug_dict = {key.lower(): value.lower() for key, value in n_type_drug_dict.items()}
type_drug_dict = hl.dict(type_drug_dict)

In [ ]:
all_ht = all_ht.annotate(group=type_drug_dict.get(all_ht.drug_name.lower(), hl.missing(hl.tstr)))

In [ ]:
all_ht = all_ht.annotate(
    drug_name = hl.if_else(
        all_ht.drug_name == 'olmesartan medoxomil', 
        'olmesartan',
        all_ht.drug_name 
    )
)
all_ht = all_ht.annotate(
    drug_name = hl.if_else(
        all_ht.drug_name == 'losartan potassium', 
        'losartan',
        all_ht.drug_name 
    )
)
all_ht = all_ht.annotate(
    drug_name = hl.if_else(
        all_ht.drug_name == 'candesartan cilexetil', 
        'candesartan',
        all_ht.drug_name 
    )
)

In [ ]:
read_v2_diuretic_dict = dict(zip(read_v2_code_df['read_code'], read_v2_code_df['with_diuretic']))
read_ctv3_diuretic_dict = dict(zip(read_ctv3_code_df['read_code'], read_ctv3_code_df['with_diuretic']))
dmd_diuretic_dict = dict(zip(dmd_code_df['dmd_code'], dmd_code_df['with_diuretic']))
bnf_diuretic_dict = dict(zip(bnf_code_df['bnf_code'], bnf_code_df['with_diuretic']))

read_v2_literal = hl.literal(read_v2_diuretic_dict)
dmd_literal = hl.literal(dmd_diuretic_dict)
read_ctv3_literal = hl.literal(read_ctv3_diuretic_dict)
bnf_literal = hl.literal(bnf_diuretic_dict)

In [ ]:
all_ht = all_ht.annotate(
    with_diuretic = hl.case()
        .when((all_ht.system == 'read_2') & (read_v2_literal.contains(all_ht.code)),
              read_v2_literal[all_ht.code])
        .when((all_ht.system == 'dmd') & (dmd_literal.contains(all_ht.code)),
              dmd_literal[all_ht.code])
        .when((all_ht.system == 'read_3') & (read_ctv3_literal.contains(all_ht.code)),
              read_ctv3_literal[all_ht.code])
        .when((all_ht.system == 'bnf') & (bnf_literal.contains(all_ht.code)),
              bnf_literal[all_ht.code])
        .default(False)  
)

In [ ]:
diuretic_substrings = [key for key, value in type_drug_dict.items() if value == 'diuretic']
diuretic_substrings.append('diuretic')
all_ht = all_ht.annotate(
    with_diuretic = hl.case()
        .when((all_ht.with_diuretic == False) 
              & (hl.any(lambda substring: all_ht['info'].lower().contains(substring), diuretic_substrings)),
              True
        ).default(all_ht.with_diuretic)
)

In [ ]:
read_v2_ccb_dict = dict(zip(read_v2_code_df['read_code'], read_v2_code_df['with_calcium_channel_blocker']))
read_ctv3_ccb_dict = dict(zip(read_ctv3_code_df['read_code'], read_ctv3_code_df['with_calcium_channel_blocker']))
dmd_ccb_dict = dict(zip(dmd_code_df['dmd_code'], dmd_code_df['with_calcium_channel_blocker']))
bnf_ccb_dict = dict(zip(bnf_code_df['bnf_code'], bnf_code_df['with_calcium_channel_blocker']))

read_v2_literal = hl.literal(read_v2_ccb_dict)
dmd_literal = hl.literal(dmd_ccb_dict)
read_ctv3_literal = hl.literal(read_ctv3_ccb_dict)
bnf_literal = hl.literal(bnf_ccb_dict)

In [ ]:
all_ht = all_ht.annotate(
    with_calcium_channel_blocker = hl.case()
        .when((all_ht.system == 'read_2') & (read_v2_literal.contains(all_ht.code)),
              read_v2_literal[all_ht.code])
        .when((all_ht.system == 'dmd') & (dmd_literal.contains(all_ht.code)),
              dmd_literal[all_ht.code])
        .when((all_ht.system == 'read_3') & (read_ctv3_literal.contains(all_ht.code)),
              read_ctv3_literal[all_ht.code])
        .when((all_ht.system == 'bnf') & (bnf_literal.contains(all_ht.code)),
              bnf_literal[all_ht.code])
        .default(False)  
)

In [ ]:
ccb_substrings = [key for key, value in type_drug_dict.items() if value == 'calcium-channel blocker']
ccb_substrings.append('calcium channel blocker')
ccb_substrings.append('calcium-channel blocker')
all_ht = all_ht.annotate(
    with_calcium_channel_blocker = hl.case()
        .when((all_ht.with_calcium_channel_blocker == False) 
              & (hl.any(lambda substring: all_ht['info'].lower().contains(substring), ccb_substrings)),
              True
        ).default(all_ht.with_calcium_channel_blocker)
)

In [ ]:
all_ht.describe()

In [ ]:
db_name = "arb_db"
full_tb_name = "all_presc.ht"

stmt = f"CREATE DATABASE IF NOT EXISTS {db_name} LOCATION 'dnax://'"
print(stmt)

spark.sql(stmt).show()

In [ ]:
db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database")['id']
url = f"dnax://{db_uri}/{full_tb_name}"

In [ ]:
all_ht.write(url, overwrite=True)